# Kaggle: Phi-3 Socratic tutor LoRA (one run)

Fine-tune [microsoft/Phi-3-mini-4k-instruct](https://huggingface.co/microsoft/Phi-3-mini-4k-instruct) on `socratic_train.jsonl`, log ScienceQA to **W&B** after each epoch, upload **final LoRA only** to Hugging Face.

**Why W&B was empty on the MSI:** Windows often hangs in `wandb.login()` / `wandb.init()`, so **no cloud run is created**. Charts only appear if you see a `W&B run: https://wandb.ai/...` line. Kaggle is Linux + internet, so this path usually works.

## Kaggle session settings (required)

1. **Accelerator:** GPU T4 (or P100). Not CPU.
2. **Internet:** On.
3. **Add-ons → Secrets** (never paste keys in cells):
   - `WANDB_API_KEY`
   - `HF_TOKEN` (write token)
4. W&B project: `socratic-phi3` → [wandb.ai](https://wandb.ai) after the run URL prints.

Default below is **1 epoch** so a session can finish. Set `EPOCHS = 3` for the full job.

**Checkpoints:** Kaggle has no Google Drive. `/kaggle/working` is wiped when the session dies unless you **Save Version**. This notebook writes checkpoints there **and** uploads the **final** LoRA to Hugging Face so a new session can continue from Hub (`--download-checkpoints`) or from a saved working directory.

In [ ]:
import os, subprocess, sys

REPO = "https://github.com/Sushey01/Socratic-Model-Fine-Tune.git"
HF_HUB_REPO = "Susu11/socratic-phi3"
WANDB_PROJECT = "socratic-phi3"
EPOCHS = 1  # change to 3 for a full train
OUTPUT_DIR = "/kaggle/working/socratic_finetuned_model"  # gone when the session ends unless you Save Version

assert os.path.exists("/kaggle/input") or True
print("python", sys.version.split()[0])

import torch
print("torch", torch.__version__, "cuda", torch.cuda.is_available(), torch.cuda.get_device_name(0) if torch.cuda.is_available() else "NO GPU")
if not torch.cuda.is_available():
    raise SystemExit("Turn on GPU T4/P100 in the Kaggle notebook settings, then Restart session.")

In [ ]:
from kaggle_secrets import UserSecretsClient

sec = UserSecretsClient()
os.environ["WANDB_API_KEY"] = sec.get_secret("WANDB_API_KEY").strip()
os.environ["HF_TOKEN"] = sec.get_secret("HF_TOKEN").strip()
os.environ["HUGGING_FACE_HUB_TOKEN"] = os.environ["HF_TOKEN"]
os.environ["HF_HUB_REPO"] = HF_HUB_REPO
os.environ["WANDB_PROJECT"] = WANDB_PROJECT
os.environ["WANDB_START_METHOD"] = "thread"
print("Secrets loaded (values not printed).")

Do **not** `pip install torch` here (Kaggle already has CUDA torch). Do **not** `pip install wandb[sandbox]` or `wandb login`.

In [ ]:
%pip install -q -U peft trl bitsandbytes datasets transformers accelerate huggingface_hub wandb python-dotenv

In [ ]:
import os
from pathlib import Path

ROOT = Path("/kaggle/working/Socratic-Model-Fine-Tune")
if not ROOT.exists():
    subprocess.check_call(["git", "clone", "--depth", "1", REPO, str(ROOT)])
else:
    subprocess.check_call(["git", "-C", str(ROOT), "pull"])
os.chdir(ROOT)
print("repo", ROOT, "jsonl", (ROOT / "socratic_train.jsonl").is_file())

In [ ]:
cmd = [
    sys.executable, "train.py",
    "--epochs", str(EPOCHS),
    "--output-dir", OUTPUT_DIR,
    "--push-to-hub", HF_HUB_REPO,
]
print("+", " ".join(cmd), flush=True)
print("Expect: W&B run: https://... then [train] heartbeat / loss lines, then ScienceQA after each epoch.", flush=True)
subprocess.check_call(cmd)

If training finished: adapters are in `/kaggle/working/Socratic-Model-Fine-Tune/socratic_finetuned_model` and on Hugging Face if `HF_TOKEN` can write. Open the printed W&B URL for loss + `eval/scienceqa_acc` / `eval/scienceqa_sri`.

Kaggle output is wiped when the session ends unless you **Save Version** or the Hub push succeeded.